# Example 10.1.3 / 3: When Coupling Breaks the Network Structure

Companion notebook for **10.1.3 Multicommodity Flows and Side Constraints** in the AMPL lab Documentation, section "When Coupling Breaks the Network Structure". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## When Coupling Breaks the Network Structure

`A` goes from `x1` to `z2`, `B` from `x1` to `y2`, and `C` from `y1` to `z2`. The arc is declared `integer`, and `option relax_integrality 1` drops that requirement for the first solve:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set NODES;  # network nodes
set LINKS within NODES cross NODES;  # directed shipping links
set PRODS;  # commodities sharing the links

param supply {NODES, PRODS} >= 0 default 0;   # units available
param demand {NODES, PRODS} >= 0 default 0;   # units required
param cost {LINKS} >= 0;                      # cost per unit, any product
param cap_joint {LINKS} >= 0;                 # limit for all products together

minimize TotalCost;  # terms are added by the arc declarations

node Balance {k in NODES, p in PRODS}:  # per-product balance at each node
    net_in = demand[k, p] - supply[k, p];

# whole units of product p on link (i, j)
arc Ship {(i, j) in LINKS, p in PRODS} >= 0 integer,
    from Balance[i, p], to Balance[j, p],
    obj TotalCost cost[i, j];

subject to JointCapacity {(i, j) in LINKS}:  # side constraint: all products share the link
    sum {p in PRODS} Ship[i, j, p] <= cap_joint[i, j];

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
lanes = [("x1", "x2"), ("y1", "y2"), ("z1", "z2")]
connectors = [("x2", "y1"), ("x2", "z1"), ("y2", "z1")]
direct = [("x1", "y2"), ("x1", "z2"), ("y1", "z2")]
ampl.set["NODES"] = ["x1", "x2", "y1", "y2", "z1", "z2"]
ampl.set["LINKS"] = lanes + connectors + direct
ampl.set["PRODS"] = ["A", "B", "C"]
ampl.param["supply"] = {("x1", "A"): 1, ("x1", "B"): 1, ("y1", "C"): 1}
ampl.param["demand"] = {("z2", "A"): 1, ("y2", "B"): 1, ("z2", "C"): 1}
ampl.param["cost"] = {link: 1 for link in lanes} | {link: 0 for link in connectors} | {link: 5 for link in direct}
ampl.param["cap_joint"] = {link: 1 for link in lanes} | {link: 3 for link in connectors + direct}

In [5]:
# run.py
ampl.eval("expand Ship['x1', 'x2', 'A'];")
ampl.option["relax_integrality"] = 1
ampl.solve(solver="highs")
ship = ampl.get_data("Ship").to_dict()
print({key: value for key, value in ship.items() if value > 0})
ampl.option["relax_integrality"] = 0
ampl.solve(solver="highs")
ship = ampl.get_data("Ship").to_dict()
print({key: value for key, value in ship.items() if value > 0})

Coefficients of Ship['x1','x2','A']:
	Balance['x1','A']         -1
	Balance['x2','A']          1
	JointCapacity['x1','x2']   1
	TotalCost                  1

HiGHS 1.14.0: optimal solution; objective 10.5
4 simplex iterations
0 barrier iterations
{('x1', 'x2', 'A'): 0.5, ('x1', 'x2', 'B'): 0.5, ('x1', 'y2', 'B'): 0.5, ('x1', 'z2', 'A'): 0.5, ('x2', 'y1', 'B'): 0.5, ('x2', 'z1', 'A'): 0.5, ('y1', 'y2', 'B'): 0.5, ('y1', 'y2', 'C'): 0.5, ('y1', 'z2', 'C'): 0.5, ('y2', 'z1', 'C'): 0.5, ('z1', 'z2', 'A'): 0.5, ('z1', 'z2', 'C'): 0.5}
HiGHS 1.14.0: optimal solution; objective 12
0 simplex iterations
0 branching nodes
{('x1', 'x2', 'B'): 1, ('x1', 'z2', 'A'): 1, ('x2', 'y1', 'B'): 1, ('y1', 'y2', 'B'): 1, ('y1', 'z2', 'C'): 1}


### Pure AMPL: data.dat and model.run

The same in pure AMPL:

In [6]:
%%writefile data.dat
set NODES := x1 x2 y1 y2 z1 z2;
set LINKS := (x1,x2) (y1,y2) (z1,z2) (x2,y1) (x2,z1) (y2,z1) (x1,y2) (x1,z2) (y1,z2);
set PRODS := A B C;
param supply := [x1,A] 1 [x1,B] 1 [y1,C] 1;
param demand := [z2,A] 1 [y2,B] 1 [z2,C] 1;
param cost := [x1,x2] 1 [y1,y2] 1 [z1,z2] 1 [x2,y1] 0 [x2,z1] 0 [y2,z1] 0
    [x1,y2] 5 [x1,z2] 5 [y1,z2] 5;
param cap_joint default 3 := [x1,x2] 1 [y1,y2] 1 [z1,z2] 1;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
expand Ship['x1', 'x2', 'A'];
option relax_integrality 1;
solve;
display {(i, j) in LINKS, p in PRODS: Ship[i, j, p] > 0} Ship[i, j, p];
option relax_integrality 0;
solve;
display {(i, j) in LINKS, p in PRODS: Ship[i, j, p] > 0} Ship[i, j, p];

Coefficients of Ship['x1','x2','A']:
	Balance['x1','A']         -1
	Balance['x2','A']          1
	JointCapacity['x1','x2']   1
	TotalCost                  1

HiGHS 1.14.0: optimal solution; objective 10.5
4 simplex iterations
0 barrier iterations
Ship[i,j,p] :=
x1 x2 A   0.5
x1 x2 B   0.5
x1 y2 B   0.5
x1 z2 A   0.5
x2 y1 B   0.5
x2 z1 A   0.5
y1 y2 B   0.5
y1 y2 C   0.5
y1 z2 C   0.5
y2 z1 C   0.5
z1 z2 A   0.5
z1 z2 C   0.5
;

HiGHS 1.14.0: optimal solution; objective 12
0 simplex iterations
0 branching nodes
Ship[i,j,p] :=
x1 x2 B   1
x1 z2 A   1
x2 y1 B   1
y1 y2 B   1
y1 z2 C   1
;

